# Etapa 1: Importação das bibliotecas

In [ ]:
import numpy as np
import math
import re
import pandas as pd
from bs4 import BeautifulSoup
from google.colab import drive
import zipfile
import seaborn as sns
import spacy as sp
import string
import random
import matplotlib.pyplot as plt

In [ ]:
%tensorflow_version 2.x
import tensorflow as tf
tf.__version__

Colab only includes TensorFlow 2.x; %tensorflow_version has no effect.


'2.12.0'

In [ ]:
from tensorflow.keras import layers
import tensorflow_datasets as tfds

# Etapa 2: Pré-processamento dos dados

## Carregamento dos arquivos

In [ ]:
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
path = '/content/drive/MyDrive/trainingandtestdata.zip'
zip_object = zipfile.ZipFile(file = path, mode = 'r')
zip_object.extractall('./')
zip_object.close()

In [ ]:
cols = ['sentiment', 'id', 'date', 'query', 'user', 'text']

In [ ]:
train_data = pd.read_csv('/content/trainingandtestdata/train.csv', header = None,
                         names = cols, engine = 'python', encoding = 'latin1')

In [ ]:
train_data.shape

(1600000, 6)

In [ ]:
train_data.head()

,sentiment,id,date,query,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [ ]:
train_data.tail()

,sentiment,id,date,query,user,text
1599995,4,2193601966,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,AmandaMarie1028,Just woke up. Having no school is the best fee...
1599996,4,2193601969,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,TheWDBoards,TheWDB.com - Very cool to hear old Walt interv...
1599997,4,2193601991,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,bpbabe,Are you ready for your MoJo Makeover? Ask me f...
1599998,4,2193602064,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,tinydiamondz,Happy 38th Birthday to my boo of alll time!!! ...
1599999,4,2193602129,Tue Jun 16 08:40:50 PDT 2009,NO_QUERY,RyanTrevMorris,happy #charitytuesday @theNSPCC @SparksCharity...


In [ ]:
train_data.sentiment.unique()

array([0, 4])

In [ ]:
sns.countplot(train_data.sentiment);

In [ ]:
test_data = pd.read_csv("/content/trainingandtestdata/test.csv",
    header=None,
    names=cols,
    engine="python",
    encoding="latin1"
)

In [ ]:
test_data.head()

,sentiment,id,date,query,user,text
0,4,3,Mon May 11 03:17:40 UTC 2009,kindle2,tpryan,@stellargirl I loooooooovvvvvveee my Kindle2. ...
1,4,4,Mon May 11 03:18:03 UTC 2009,kindle2,vcu451,Reading my kindle2... Love it... Lee childs i...
2,4,5,Mon May 11 03:18:54 UTC 2009,kindle2,chadfu,"Ok, first assesment of the #kindle2 ...it fuck..."
3,4,6,Mon May 11 03:19:04 UTC 2009,kindle2,SIX15,@kenburbary You'll love your Kindle2. I've had...
4,4,7,Mon May 11 03:21:41 UTC 2009,kindle2,yamarama,@mikefish Fair enough. But i have the Kindle2...


In [ ]:
test_data.shape

(498, 6)

In [ ]:
test_data.sentiment.unique()

array([4, 0, 2])

In [ ]:
sns.countplot(test_data.sentiment);

## Pré-processamento

### Limpeza

In [ ]:
data = train_data

In [ ]:
data.drop(['id', 'date', 'query', 'user'], axis = 1, inplace=True)

In [ ]:
data.head()

,sentiment,text
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,is upset that he can't update his Facebook by ...
2,0,@Kenichan I dived many times for the ball. Man...
3,0,my whole body feels itchy and like its on fire
4,0,"@nationwideclass no, it's not behaving at all...."


In [ ]:
previsores = data.iloc[:, 1].values

In [ ]:
previsores

array(["@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer.  You shoulda got David Carr of Third Day to do it. ;D",
       "is upset that he can't update his Facebook by texting it... and might cry as a result  School today also. Blah!",
       '@Kenichan I dived many times for the ball. Managed to save 50%  The rest go out of bounds',
       ..., 'Are you ready for your MoJo Makeover? Ask me for details ',
       'Happy 38th Birthday to my boo of alll time!!! Tupac Amaru Shakur ',
       'happy #charitytuesday @theNSPCC @SparksCharity @SpeakingUpH4H '],
      dtype=object)

In [ ]:
previsores.shape

(1600000,)

In [ ]:
type(previsores)

numpy.ndarray

In [ ]:
classe = data.iloc[:, 0].values
classe

array([0, 0, 0, ..., 4, 4, 4])

In [ ]:
from sklearn.model_selection import train_test_split
previsores, _, classe, _ = train_test_split(previsores, classe, test_size = 0.85, stratify = classe)

In [ ]:
previsores.shape

(240000,)

In [ ]:
classe.shape

(240000,)

In [ ]:
unique, counts = np.unique(classe, return_counts=True)
unique, counts

(array([0, 4]), array([120000, 120000]))

In [ ]:
def clean_tweets(tweet):
  tweet = BeautifulSoup(tweet, 'lxml').get_text()
  tweet = re.sub(r"@[A-Za-z0-9]+", ' ', tweet)
  tweet = re.sub(r"https?://[A-Za-z0-9./]+", ' ', tweet)
  tweet = re.sub(r"[^a-zA-Z.!?]", ' ', tweet)
  tweet = re.sub(r" +", ' ', tweet)

  tweet = tweet.lower()
  document = nlp(tweet)

  words = []
  for token in document:
    words.append(token.text)

  words = [word for word in words if word not in stop_words and word not in string.punctuation]
  words = ' '.join([str(element) for element in words])

  return words

In [ ]:
text = "@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer.  2 You shoulda got David Carr of Third Day to do it. ;D"

In [ ]:
text = clean_tweets(text)
text

<ipython-input-36-cb97bb38e0ad>:2: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  tweet = BeautifulSoup(tweet, 'lxml').get_text()


'  awww s bummer shoulda got david carr day d'

In [ ]:
nlp = sp.load("en_core_web_sm")
nlp

In [ ]:
stop_words = sp.lang.en.STOP_WORDS

In [ ]:
print(stop_words)

{'less', 'beforehand', 'name', 'besides', 'any', 'three', 'except', 'take', 'wherever', 'top', 'among', 'than', 'it', 'give', 'those', 'that', 'front', 'even', 'though', 'noone', 'therein', 'throughout', 'their', 'some', 'whose', 'everywhere', 'after', 'anything', 'rather', 'is', 'meanwhile', 'toward', 'four', 'neither', 'become', '’s', 'since', 'side', "n't", 'empty', 'nor', 'other', 'used', 'amongst', 'me', 'would', 'just', 'no', '’d', "'ve", 'move', 'as', 'whoever', 'am', 'which', 'only', 'moreover', 'eight', 'whereupon', 'formerly', 'do', 'please', 'everyone', 'also', 'with', 'hereafter', 'itself', 'they', 'whereas', 'before', 'anyone', 'many', 'does', 'becoming', 'full', 'down', 'are', 'may', 'beside', 'at', 'two', 'wherein', 'she', 'when', 'up', '‘s', 'much', 'thereby', 'myself', 'sometime', 'none', 'about', 'hereby', 'been', 'had', 'done', '‘re', 'ourselves', 'thence', 'around', 'along', 'he', 'put', 'using', 'hence', 'anyhow', 'often', 'can', 'make', 'upon', 'why', 'latter', 'f

In [ ]:
len(stop_words)

326

In [ ]:
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [ ]:
data_clean = [clean_tweets(tweet) for tweet in previsores]

<ipython-input-36-cb97bb38e0ad>:2: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  tweet = BeautifulSoup(tweet, 'lxml').get_text()


In [ ]:
for _ in range(10):
  print(data_clean[random.randint(0, len(data_clean) - 1)])

  working xserver right s playing nicely suddenly afp logs aren t helpful rebuild
changing leslie la trip m sad
  mannnnnn missed need international time session haha .... damn
  guess s fault w blanka
  know ... thought d spilling beans ... murderer
mmm crunchy bran strawberries love breakfast
talk peoples s everybody today
  good morning rita
  don t forget save complete set sent email reserve
  plenty strip


In [ ]:
data_labels = classe

In [ ]:
data_labels[data_labels == 4] = 1

In [ ]:
data_labels

array([1, 0, 0, ..., 0, 1, 1])

In [ ]:
np.unique(data_labels)

array([0, 1])

### Tokenização

In [ ]:
#tokenizer = tfds.features.text.SubwordTextEncoder.build_from_corpus(data_clean, target_vocab_size=2**16)
tokenizer = tfds.deprecated.text.SubwordTextEncoder.build_from_corpus(data_clean, target_vocab_size=2**16)

In [ ]:
tokenizer.vocab_size

52804

In [ ]:
print(tokenizer.subwords)

In [ ]:
ids = tokenizer.encode('i am happy')
ids

[281, 5022, 374]

In [ ]:
text = tokenizer.decode(ids)
text

'i am happy'

In [ ]:
data_inputs = [tokenizer.encode(sentence) for sentence in data_clean]

In [ ]:
for _ in range(10):
  print(data_inputs[random.randint(0, len(data_inputs) - 1)])

[1, 14, 4, 978, 1123, 157, 785, 11013, 50296, 64, 2158, 38643, 52580, 6450, 578]
[45, 96, 81, 6474, 605, 115, 5663, 8173]
[1, 1245, 359, 30980, 1458, 6235, 4, 3756, 543, 4, 978, 42029, 52664]
[2256, 231, 22776]
[1, 3966, 17123, 52663, 5, 1721, 1135, 482]
[530, 6, 6, 4366, 28813, 6, 3, 344]
[133, 340, 151, 21, 710, 10006, 3150, 8514, 2310, 589, 6318, 24539, 14943]
[3, 5946, 1725, 542, 439, 2811]
[1, 5510, 63, 5, 515, 426, 12842, 3, 2748, 1289, 2598, 8420, 444]
[1, 28967, 134, 8, 60, 31, 222, 159, 1652, 1435, 530, 28, 173, 116]


### Padding

In [ ]:
max_len = max([len(sentence) for sentence in data_inputs])
max_len

59

In [ ]:
data_inputs = tf.keras.preprocessing.sequence.pad_sequences(data_inputs,
                                                            value = 0,
                                                            padding = 'post',
                                                            maxlen=max_len)

In [ ]:
for _ in range(10):
  print(data_inputs[random.randint(0, len(data_inputs) - 1)])

[462 723  17  18 246 320  12  90   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0]
[    1   261    10  1658   560 26907  3190  1838 35613     2    12 10099
  1535     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0]
[329   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0]
[   11    17   128 40108     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0

### Divisão da base de dados em treinamento e teste



In [ ]:
train_inputs, test_inputs, train_labels, test_labels = train_test_split(data_inputs,
                                                                        data_labels,
                                                                        test_size=0.3,
                                                                        stratify = data_labels)

In [ ]:
train_inputs[0]

array([   68,   193,   228,   293, 29759,  3664,   223,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     0,     0], dtype=int32)

In [ ]:
train_inputs.shape

(168000, 59)

In [ ]:
train_labels.shape

(168000,)

In [ ]:
test_inputs.shape

(72000, 59)

In [ ]:
test_labels.shape

(72000,)

# Etapa 3: Construção do modelo

In [ ]:
class DCNN(tf.keras.Model):

  def __init__(self,
               vocab_size,
               emb_dim=128,
               nb_filters=50,
               ffn_units=512,
               nb_classes=2,
               dropout_rate=0.1,
               training=True,
               name="dcnn"):
    super(DCNN, self).__init__(name=name)

    self.embedding = layers.Embedding(vocab_size, emb_dim)

    self.bigram = layers.Conv1D(filters=nb_filters, kernel_size=2, padding='same', activation='relu')

    self.trigram = layers.Conv1D(filters=nb_filters, kernel_size=3, padding='same', activation='relu')

    self.fourgram = layers.Conv1D(filters=nb_filters, kernel_size=4, padding='same', activation='relu')

    self.pool = layers.GlobalMaxPool1D()

    self.dense_1 = layers.Dense(units = ffn_units, activation = 'relu')

    self.dropout = layers.Dropout(rate = dropout_rate)
    if nb_classes == 2:
      self.last_dense = layers.Dense(units = 1, activation = 'sigmoid')
    else:
      self.last_dense = layers.Dense(units = nb_classes, activation = 'softmax')

  def call(self, inputs, training):
    x = self.embedding(inputs)
    x_1 = self.bigram(x)
    x_1 = self.pool(x_1)
    x_2 = self.trigram(x)
    x_2 = self.pool(x_2)
    x_3 = self.fourgram(x)
    x_3 = self.pool(x_3)

    merged = tf.concat([x_1, x_2, x_3], axis = -1) # (batch_size, 3 * nb_filters)
    merged = self.dense_1(merged)
    merged = self.dropout(merged, training)
    output = self.last_dense(merged)

    return output

# Etapa 4: Treinamento

## Configuração dos parâmetros

In [ ]:
vocab_size = tokenizer.vocab_size
vocab_size

52804

In [ ]:
emb_dim = 200
nb_filters = 100
ffn_units = 256
batch_size = 64
nb_classes = len(set(train_labels))
dropout_rate = 0.2
nb_epochs = 5

## Treinamento

In [ ]:
Dcnn = DCNN(vocab_size=vocab_size, emb_dim=emb_dim, nb_filters=nb_filters,
            ffn_units=ffn_units, nb_classes=nb_classes, dropout_rate=dropout_rate)

In [ ]:
if nb_classes == 2:
  Dcnn.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
else:
  Dcnn.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
checkpoint_path = "./"
ckpt = tf.train.Checkpoint(Dcnn=Dcnn)
ckpt_manager = tf.train.CheckpointManager(ckpt, checkpoint_path, max_to_keep=5)
if ckpt_manager.latest_checkpoint:
  ckpt.restore(ckpt_manager.latest_checkpoint)
  print('Latest checkpoint restored')

In [ ]:
history = Dcnn.fit(train_inputs, train_labels,
                   batch_size = batch_size,
                   epochs = nb_epochs,
                   verbose = 1,
                   validation_split = 0.10)
ckpt_manager.save()

Epoch 1/5
2363/2363 [==============================] - 1170s 493ms/step - loss: 0.5024 - accuracy: 0.7513 - val_loss: 0.4826 - val_accuracy: 0.7622
Epoch 2/5
2363/2363 [==============================] - 1135s 480ms/step - loss: 0.3626 - accuracy: 0.8388 - val_loss: 0.5314 - val_accuracy: 0.7513
Epoch 3/5
2363/2363 [==============================] - 1121s 474ms/step - loss: 0.1935 - accuracy: 0.9190 - val_loss: 0.7387 - val_accuracy: 0.7310
Epoch 4/5
2363/2363 [==============================] - 1123s 475ms/step - loss: 0.1024 - accuracy: 0.9578 - val_loss: 1.0422 - val_accuracy: 0.7301
Epoch 5/5
2363/2363 [==============================] - 1102s 466ms/step - loss: 0.0689 - accuracy: 0.9708 - val_loss: 1.3289 - val_accuracy: 0.7292


'./ckpt-1'

# Etapa 5: Avaliação do modelo

## Avaliação

In [ ]:
results = Dcnn.evaluate(test_inputs, test_labels, batch_size=batch_size)
print(results)

1125/1125 [==============================] - 45s 40ms/step - loss: 1.2961 - accuracy: 0.7347
[1.2961052656173706, 0.7347360849380493]


In [ ]:
y_pred_test = Dcnn.predict(test_inputs)

2250/2250 [==============================] - 58s 25ms/step


In [ ]:
y_pred_test

array([[0.99507654],
       [0.5718968 ],
       [0.3339544 ],
       ...,
       [0.0032403 ],
       [0.9991948 ],
       [0.6851953 ]], dtype=float32)

In [ ]:
y_pred_test = (y_pred_test > 0.5)

In [ ]:
y_pred_test

array([[ True],
       [ True],
       [False],
       ...,
       [False],
       [ True],
       [ True]])

In [ ]:
test_labels

array([1, 1, 0, ..., 0, 1, 1])

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(test_labels, y_pred_test)
cm

array([[27211,  8789],
       [10310, 25690]])

In [ ]:
sns.heatmap(cm, annot=True)

In [ ]:
history.history.keys()

dict_keys(['loss', 'accuracy', 'val_loss', 'val_accuracy'])

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss progress during training and validation')
plt.xlabel('Epoch')
plt.ylabel('Losses')
plt.legend(['Training loss', 'Validation loss'])

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model accuracy progress during training and validation')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(['Training accuracy', 'Validation accuracy'])

## Previsões

In [ ]:
text = 'i love you'
text = tokenizer.encode(text)
text

[281, 13, 3564]

In [ ]:
Dcnn(np.array([text]), training=False).numpy()

array([[0.608221]], dtype=float32)